<a href="https://colab.research.google.com/github/cem8kaya/5g-ai-lab/blob/main/5G_AI_LAB_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5G AI/ML Laboratory - kayacem.com

## 1. Overview
This environment is an end-to-end simulated 5G Network designed for generating telecommunications traffic data for Machine Learning and Data Science projects. It utilizes Open5GS for the Core Network and UERANSIM for the RAN/UE simulation.



## 2. Infrastructure Details
* **Platform:** Google Cloud Platform (GCP)
* **OS:** Ubuntu 22.04 LTS (Kernel: 6.8.0-1048-gcp)
* **Execution Environment:** Google Colab

## 3. 5G Core Network (Open5GS)
* **Services:** AMF, UPF, SMF, UDM, AUSF, etc. (Running as `systemd` services, e.g., `open5gs-amfd`).
* **Kernel Module:** `gtp5g` is compiled and loaded successfully for UPF packet routing.
* **WebUI:** Accessible externally via port `9999` (e.g., `http://<EXTERNAL_IP>:9999`).
* **Default Network Configuration:** * PLMN: `999-70` (MCC: 999, MNC: 70)

## 4. RAN & UE Simulation (UERANSIM)
* **Installation Path:** Compiled from source, binaries located at `/opt/UERANSIM/build/`.
* **Configuration Files:** * gNodeB: `/opt/UERANSIM/config/open5gs-gnb.yaml` (Configured for PLMN 999-70)
    * UE: `/opt/UERANSIM/config/open5gs-ue.yaml` (Configured for PLMN 999-70)
* **Subscriber Details (Registered in Open5GS WebUI):**
    * **IMSI:** `999700000000001`
* **Tunnel Interface:** When the UE establishes a PDU session, it creates a virtual network interface named `uesimtun0`.

## 5. Data Science Pipeline (ETL)
* **Traffic Generation:** Synthetic traffic (ICMP, TCP, HTTP) is generated on the GCP VM using `ping` and `curl` bound to the `uesimtun0` interface.
* **Data Capture:** `tcpdump` captures the raw packets into a `.pcap` file (`/tmp/5g_traffic.pcap`).
* **Data Processing:** The `.pcap` file is transferred to Google Colab, where the Python `scapy` library parses it.
* **Current State:** A Pandas DataFrame is active with features including `Zaman` (Timestamp), `Kaynak_IP`, `Hedef_IP`, `Protokol`, `Boyut_Byte`, `Zaman_Farki_sn` (Inter-arrival time), and `Yon` (Uplink/Downlink).

# Hücre 0: Bağımlılıklar

In [2]:
# 1. GCP'ye Kimlik Doğrulaması
from google.colab import auth
auth.authenticate_user()
print("Kimlik doğrulaması başarılı!")

Kimlik doğrulaması başarılı!


# Hücre 1: Konfigürasyon

In [ ]:
PROJECT_ID = "g-ai-lab-491619"
ZONE       = "europe-west4-a"
VM_NAME    = "open5gs-ai-lab"
UE_IP      = "10.45.0.3"
UE_IFACE   = "uesimtun0"
NR_BINDER  = "/opt/UERANSIM/build/nr-binder"

print(f"✅ Konfigürasyon yüklendi — VM: {VM_NAME} | UE: {UE_IP}")

✅ Konfigürasyon yüklendi — VM: open5gs-ai-lab | UE: 10.45.0.3


In [1]:
# =============================================================================
# 5G AI/ML LABORATORY — GOOGLE COLAB NOTEBOOK
# Open5GS + UERANSIM tabanlı 5G Core Network Simülasyon ve ML Pipeline
# =============================================================================
# Geliştirici  : 5G AI/ML Lab
# Platform     : Google Colab + GCP (europe-west4-a)
# VM           : open5gs-ai-lab (Ubuntu 22.04, Open5GS v2.7.6)
# Son Güncell. : 2026-03 (v2 — tüm düzeltmeler dahil)
# =============================================================================

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 0 — BAĞIMLILIK KURULUMU                                             ║
# ╠══════════════════════════════════════════════════════════════════════════════╣
# ║  Açıklama:                                                                  ║
# ║    Notebook boyunca kullanılacak Python kütüphanelerini kurar.              ║
# ║    • pandas   : CSV/DataFrame işlemleri ve ETL pipeline                    ║
# ║    • scapy    : PCAP okuma/yazma (ileride GTP-U analizi için)              ║
# ║    • matplotlib/seaborn : Görselleştirme katmanı                           ║
# ║    • numpy    : Sayısal hesaplama (feature engineering)                    ║
# ║  Not: Colab ortamında her runtime yenilenmesinde bir kez çalıştırılmalı.   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'pandas', 'scapy', 'matplotlib', 'seaborn', 'numpy'])
print("✅ Tüm bağımlılıklar hazır.")

✅ Tüm bağımlılıklar hazır.


In [3]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 1 — MERKEZİ KONFİGÜRASYON                                          ║
# ╠══════════════════════════════════════════════════════════════════════════════╣
# ║  Açıklama:                                                                  ║
# ║    Tüm notebook boyunca kullanılan GCP ve 5G parametrelerini tanımlar.     ║
# ║    • PROJECT_ID / ZONE / VM_NAME : GCP kaynak tanımlayıcıları             ║
# ║    • UE_IP / UE_IFACE            : UERANSIM UE tünel arayüzü              ║
# ║    • NR_BINDER                   : GTP-U üzerinden trafik bağlama aracı   ║
# ║  Not: gcloud CLI Colab'da authenticate edilmiş olmalıdır.                  ║
# ║    → !gcloud auth login                                                     ║
# ║    → !gcloud config set project g-ai-lab-491619                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

PROJECT_ID = "g-ai-lab-491619"
ZONE       = "europe-west4-a"
VM_NAME    = "open5gs-ai-lab"
UE_IP      = "10.45.0.3"
UE_IFACE   = "uesimtun0"
NR_BINDER  = "/opt/UERANSIM/build/nr-binder"

print(f"✅ Konfigürasyon yüklendi")
print(f"   VM      : {VM_NAME}")
print(f"   Bölge   : {ZONE}")
print(f"   UE IP   : {UE_IP}")
print(f"   Arayüz  : {UE_IFACE}")


✅ Konfigürasyon yüklendi
   VM      : open5gs-ai-lab
   Bölge   : europe-west4-a
   UE IP   : 10.45.0.3
   Arayüz  : uesimtun0


In [4]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 2 — 5G CORE SAĞLIK KONTROLÜ                                         ║
# ╠══════════════════════════════════════════════════════════════════════════════╣
# ║  Açıklama:                                                                  ║
# ║    GCP VM üzerindeki 5G Core bileşenlerinin durumunu doğrular.             ║
# ║    Beklenen çıktı: 7/7 active, uesimtunX IP=10.45.0.x, upf_sessionnbr≥1  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
    echo '=== 5G Core Servisleri ==='
    for svc in amfd smfd upfd ausfd udmd pcfd nrfd; do
        STATUS=\$(systemctl is-active open5gs-\$svc 2>/dev/null)
        printf '  %-12s: %s\n' \$svc \$STATUS
    done
    echo ''
    echo '=== UERANSIM ==='
    pgrep -a nr-gnb && echo 'gNB: calisiyor' || echo 'gNB: DURDU'
    pgrep -a nr-ue  && echo 'UE : calisiyor' || echo 'UE : DURDU'
    echo ''
    echo '=== Aktif PDU Sessionlar ==='
    ip a | grep -A2 'uesimtun'
    echo ''
    echo '=== UPF Session Sayisi ==='
    curl -s http://127.0.0.7:9090/metrics 2>/dev/null | \
        grep 'upf_sessionnbr' | grep -v '#'
"

Generating public/private rsa key pair.
Your identification has been saved in /root/.ssh/google_compute_engine
Your public key has been saved in /root/.ssh/google_compute_engine.pub
The key fingerprint is:
SHA256:Xj0oEo9sIuX1xQroooytaN1SptQc/iL9RzaW6pgRf9g root@f319844db345
The key's randomart image is:
+---[RSA 3072]----+
|                 |
|     .   .       |
|    o +   o      |
|   + + * o o     |
|  o B.* S + o    |
|oo + OoooO   .   |
|o.+ *..o*E.      |
|.o = ++o..       |
|+   oo+o.        |
+----[SHA256]-----+
=== 5G Core Servisleri ===
  amfd        : active
  smfd        : active
  upfd        : active
  ausfd       : active
  udmd        : active
  pcfd        : active
  nrfd        : active

=== UERANSIM ===
42148 /opt/UERANSIM/build/nr-gnb -c /opt/UERANSIM/config/open5gs-gnb.yaml
gNB: calisiyor
78100 /opt/UERANSIM/build/nr-ue -c /opt/UERANSIM/config/open5gs-ue.yaml
UE : calisiyor

=== Aktif PDU Sessionlar ===
37: uesimtun0: <POINTOPOINT,PROMISC,NOTRAILERS,UP,LOWER_UP> mtu

This tool needs to create the directory [/root/.ssh] before being able to 
generate SSH keys.

Do you want to continue (Y/n)?  
Updating project ssh metadata...
..................................................Updated [https://www.googleapis.com/compute/v1/projects/g-ai-lab-491619].
.done.
Waiting for SSH key to propagate.


In [5]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 3 — SENARYO A: GENEL TRAFİK PROFİLİ                                ║
# ╠══════════════════════════════════════════════════════════════════════════════╣
# ║  Açıklama:                                                                  ║
# ║    uesimtun0 üzerinden ICMP/HTTP/DNS/Burst trafik üretir.                  ║
# ║    CSV: timestamp_ms, scenario, ue_ip, target, protocol, bytes, rtt_ms     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

cat > /tmp/scenario_a.sh << 'SCRIPT'
#!/bin/bash
LOG="/tmp/scenario_a.csv"
NR="/opt/UERANSIM/build"
UE="uesimtun0"
echo "timestamp_ms,scenario,ue_ip,target,protocol,bytes,rtt_ms,status" > $LOG

log() { echo "$(date +%s%3N),$1,10.45.0.3,$2,$3,$4,$5,$6" >> $LOG; }

echo "[A] ICMP — 20 ping"
for i in $(seq 1 20); do
    cd $NR
    RESULT=$(./nr-binder $UE ping -c 1 -q 8.8.8.8 2>&1)
    RTT=$(echo "$RESULT" | grep rtt | awk -F'/' '{print $5}')
    LOSS=$(echo "$RESULT" | grep transmitted | awk '{print $6}' | tr -d '%')
    STATUS=$([ "${LOSS:-100}" -eq 0 ] && echo "OK" || echo "LOSS")
    log "A_ICMP" "8.8.8.8" "ICMP" "84" "${RTT:-0}" "$STATUS"
    sleep 0.5
done

echo "[A] HTTP — web browse pattern"
for URL in \
    "http://neverssl.com" \
    "https://www.google.com" \
    "https://www.cloudflare.com" \
    "http://neverssl.com" \
    "https://www.google.com"; do
    cd $NR
    RESULT=$(./nr-binder $UE curl -s -o /dev/null \
        -w "%{size_download}|%{time_total}" \
        --max-time 15 "$URL" 2>/dev/null || echo "0|0")
    SIZE=$(echo $RESULT | cut -d'|' -f1)
    TIME=$(echo $RESULT | cut -d'|' -f2 | awk '{printf "%.0f", $1*1000}')
    STATUS=$([ "${SIZE:-0}" -gt 0 ] && echo "OK" || echo "FAIL")
    HOST=$(echo $URL | awk -F'/' '{print $3}')
    log "A_HTTP" "$HOST" "HTTP" "${SIZE:-0}" "${TIME:-0}" "$STATUS"
    sleep $(awk 'BEGIN{srand(); printf "%d", 1+rand()*3}')
done

echo "[A] DNS — latency olcumu"
for DOMAIN in google.com cloudflare.com github.com amazon.com; do
    cd $NR
    DNS_TIME=$(./nr-binder $UE curl -s -o /dev/null \
        -w "%{time_namelookup}" \
        --max-time 5 "https://$DOMAIN" 2>/dev/null || echo "0")
    DNS_MS=$(echo $DNS_TIME | awk '{printf "%.0f", $1*1000}')
    log "A_DNS" "$DOMAIN" "DNS" "0" "${DNS_MS:-0}" "OK"
done

echo "[A] Burst — 10 hizli istek (google.com)"
for i in $(seq 1 10); do
    cd $NR
    RESULT=$(./nr-binder $UE curl -s -o /dev/null \
        -w "%{size_download}|%{time_total}" \
        --max-time 10 "https://www.google.com" 2>/dev/null || echo "0|0")
    SIZE=$(echo $RESULT | cut -d'|' -f1)
    TIME=$(echo $RESULT | cut -d'|' -f2 | awk '{printf "%.0f", $1*1000}')
    STATUS=$([ "${SIZE:-0}" -gt 0 ] && echo "OK" || echo "FAIL")
    log "A_BURST" "www.google.com" "HTTP" "${SIZE:-0}" "${TIME:-0}" "$STATUS"
    sleep 1
done

echo "=== TAMAMLANDI: $(( $(wc -l < $LOG) - 1 )) kayit ==="
SCRIPT

gcloud compute scp /tmp/scenario_a.sh $VM_NAME:/tmp/scenario_a.sh \
    --project=$PROJECT_ID --zone=$ZONE
gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE \
    --command="sudo bash /tmp/scenario_a.sh"

[A] ICMP — 20 ping
[A] HTTP — web browse pattern
[A] DNS — latency olcumu
[A] Burst — 10 hizli istek (google.com)
=== TAMAMLANDI: 39 kayit ===


In [6]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 4 — SENARYO B: ETİKETLİ ANOMALİ DATASET                            ║
# ╠══════════════════════════════════════════════════════════════════════════════╣
# ║  Açıklama:                                                                  ║
# ║    5 farklı label: NORMAL / ANOMALY_SCAN / ANOMALY_FLOOD /                 ║
# ║    ANOMALY_SLOWLORIS / ANOMALY_EXFIL                                        ║
# ║    CSV: timestamp_ms, label, ue_ip, target, protocol, bytes, rtt_ms        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

cat > /tmp/scenario_b.sh << 'SCRIPT'
#!/bin/bash
LOG="/tmp/scenario_b.csv"
NR="/opt/UERANSIM/build"
UE="uesimtun0"
echo "timestamp_ms,label,ue_ip,target,protocol,bytes,rtt_ms,pkt_rate" > $LOG

log() { echo "$(date +%s%3N),$1,10.45.0.3,$2,$3,$4,$5,$6" >> $LOG; }

echo "[B] NORMAL — web browsing (15 istek)"
for i in $(seq 1 15); do
    cd $NR
    RESULT=$(./nr-binder $UE curl -s -o /dev/null \
        -w "%{size_download}|%{time_total}" \
        --max-time 10 "https://www.google.com" 2>/dev/null || echo "0|0")
    SIZE=$(echo $RESULT | cut -d'|' -f1)
    TIME=$(echo $RESULT | cut -d'|' -f2 | awk '{printf "%.0f", $1*1000}')
    log "NORMAL" "www.google.com" "HTTP" "${SIZE:-0}" "${TIME:-0}" "1"
    sleep $(awk 'BEGIN{srand('$i'); printf "%.1f", 2+rand()*4}')
done

echo "[B] NORMAL — IoT heartbeat (20 ping)"
for i in $(seq 1 20); do
    cd $NR
    RTT=$(./nr-binder $UE ping -c 1 -q 8.8.8.8 2>&1 | \
        grep rtt | awk -F'/' '{print $5}')
    log "NORMAL" "8.8.8.8" "ICMP" "84" "${RTT:-0}" "0.5"
    sleep 2
done

echo "[B] ANOMALY_SCAN — 30 paralel hizli istek"
for i in $(seq 1 30); do
    cd $NR
    (./nr-binder $UE curl -s -o /dev/null \
        --connect-timeout 1 --max-time 2 \
        "https://www.google.com" 2>/dev/null) &
    log "ANOMALY_SCAN" "www.google.com" "HTTP" "0" "0" "30"
done
wait

echo "[B] ANOMALY_FLOOD — 50 paralel ICMP"
for i in $(seq 1 50); do
    cd $NR
    (./nr-binder $UE ping -c 1 -q 8.8.8.8 > /dev/null 2>&1) &
    log "ANOMALY_FLOOD" "8.8.8.8" "ICMP" "84" "0" "50"
done
wait

echo "[B] ANOMALY_SLOWLORIS — 5x buyuk/yavas indirme"
for i in $(seq 1 5); do
    cd $NR
    RESULT=$(./nr-binder $UE curl -s -o /dev/null \
        -w "%{size_download}|%{time_total}" \
        --max-time 20 "https://www.cloudflare.com" 2>/dev/null || echo "0|0")
    SIZE=$(echo $RESULT | cut -d'|' -f1)
    TIME=$(echo $RESULT | cut -d'|' -f2 | awk '{printf "%.0f", $1*1000}')
    log "ANOMALY_SLOWLORIS" "www.cloudflare.com" "HTTP" "${SIZE:-0}" "${TIME:-0}" "0.1"
    sleep 0.5
done

echo "[B] ANOMALY_EXFIL — buyuk dosya transferi"
cd $NR
RESULT=$(./nr-binder $UE curl -s -o /dev/null \
    -w "%{size_download}|%{time_total}" \
    --max-time 30 "http://speedtest.tele2.net/1MB.zip" 2>/dev/null || echo "0|0")
SIZE=$(echo $RESULT | cut -d'|' -f1)
TIME=$(echo $RESULT | cut -d'|' -f2 | awk '{printf "%.0f", $1*1000}')
log "ANOMALY_EXFIL" "speedtest.tele2.net" "HTTP" "${SIZE:-0}" "${TIME:-0}" "0.03"

echo "=== TAMAMLANDI ==="
tail -n +2 $LOG | cut -d',' -f2 | sort | uniq -c | sort -rn
SCRIPT

gcloud compute scp /tmp/scenario_b.sh $VM_NAME:/tmp/scenario_b.sh \
    --project=$PROJECT_ID --zone=$ZONE
gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE \
    --command="sudo bash /tmp/scenario_b.sh"

[B] NORMAL — web browsing (15 istek)
[B] NORMAL — IoT heartbeat (20 ping)
[B] ANOMALY_SCAN — 30 paralel hizli istek
[B] ANOMALY_FLOOD — 50 paralel ICMP
[B] ANOMALY_SLOWLORIS — 5x buyuk/yavas indirme
[B] ANOMALY_EXFIL — buyuk dosya transferi
=== TAMAMLANDI ===
     50 ANOMALY_FLOOD
     35 NORMAL
     30 ANOMALY_SCAN
      5 ANOMALY_SLOWLORIS
      1 ANOMALY_EXFIL


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 5 — SENARYO C: UPF METRİK ZAMAN SERİSİ  [v2 - DÜZELTİLDİ]        ║
# ╠══════════════════════════════════════════════════════════════════════════════╣
# ║  Açıklama:                                                                  ║
# ║    /proc/net/dev üzerinden anlık KB/s throughput ölçer.                    ║
# ║    v2 Düzeltmeleri:                                                         ║
# ║    • bc yerine python3 ile float hesaplama (bc boş dönme hatası giderildi) ║
# ║    • 10sn warm-up: counter'ları sıfırdan kaldırır                          ║
# ║    • Dinamik interface tespiti: uesimtun0/1/2/3 otomatik seçilir           ║
# ║    • delta_ms konsol çıktısına eklendi (debug için)                        ║
# ║    Trafik fazları: normal(0-30s) → burst(30-45s) → idle(45-60s) → burst2  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"
cat > /tmp/scenario_c.sh << 'SCRIPT'
#!/bin/bash
LOG="/tmp/scenario_c.csv"
NR="/opt/UERANSIM/build"
DURATION=90
INTERVAL=3
echo "timestamp_ms,phase,ue_tx_kbps,ue_rx_kbps,ogstun_tx_kbps,ogstun_rx_kbps,ens4_tx_kbps,ens4_rx_kbps,upf_sessions" > $LOG
get_ue_iface() {
    for iface in uesimtun0 uesimtun1 uesimtun2 uesimtun3; do
        IP=$(ip a show $iface 2>/dev/null | grep 'inet ' | \
            awk '{print $2}' | cut -d'/' -f1)
        [ -n "$IP" ] && echo "$iface" && return
    done
    echo ""
}
read_bytes() {
    grep "${1}:" /proc/net/dev 2>/dev/null | \
        awk '{gsub(/:/," "); print $2, $10}'
}
calc_kbps() {
    python3 -c "
b1,b2,dt_ms = $1,$2,$3
diff = max(0, b2 - b1)
kbps = round(diff / (dt_ms/1000) / 1024, 2) if dt_ms > 0 else 0
print(kbps)
" 2>/dev/null || echo "0"
}
echo "=== UE durumu kontrol ==="
UE_IFACE=$(get_ue_iface)
if [ -z "$UE_IFACE" ]; then
    echo "  UE tunnel yok, yeniden baslatiliyor..."
    sudo pkill -f nr-ue || true
    sleep 2
    sudo nohup $NR/nr-ue \
        -c /opt/UERANSIM/config/open5gs-ue.yaml > /root/ue.log 2>&1 &
    for i in $(seq 1 40); do
        sleep 0.5
        UE_IFACE=$(get_ue_iface)
        [ -n "$UE_IFACE" ] && break
    done
fi
if [ -z "$UE_IFACE" ]; then echo "HATA: UE tunnel olusturulamadi"; exit 1; fi
UE_IP=$(ip a show $UE_IFACE | grep 'inet ' | awk '{print $2}' | cut -d'/' -f1)
echo "  Aktif: $UE_IFACE ($UE_IP)"
echo "=== Warm-up (10sn trafik) ==="
cd $NR
./nr-binder $UE_IFACE ping -c 5 8.8.8.8 > /dev/null 2>&1
./nr-binder $UE_IFACE curl -s -o /dev/null --max-time 8 \
    "https://www.google.com" > /dev/null 2>&1
sleep 2
read UE_RX1 UE_TX1 <<< $(read_bytes $UE_IFACE)
read OG_RX1 OG_TX1 <<< $(read_bytes ogstun)
read E4_RX1 E4_TX1 <<< $(read_bytes ens4)
T_PREV_MS=$(date +%s%3N)
echo "  Baslangic sayac: UE_TX=$UE_TX1 UE_RX=$UE_RX1"
(
  for i in $(seq 1 4); do
    cd $NR
    ./nr-binder $UE_IFACE ping -c 5 8.8.8.8 > /dev/null 2>&1
    ./nr-binder $UE_IFACE curl -s -o /dev/null --max-time 10 \
        "https://www.google.com" > /dev/null 2>&1
    sleep 3
  done
  for i in $(seq 1 8); do
    cd $NR
    ./nr-binder $UE_IFACE curl -s -o /dev/null --max-time 8 \
        "https://www.cloudflare.com" > /dev/null 2>&1 &
  done
  wait; sleep 2
  sleep 15
  for i in $(seq 1 5); do
    cd $NR
    ./nr-binder $UE_IFACE curl -s -o /dev/null --max-time 10 \
        "https://www.google.com" > /dev/null 2>&1
    ./nr-binder $UE_IFACE ping -c 3 8.8.8.8 > /dev/null 2>&1
    sleep 2
  done
) &
TRAFFIC_PID=$!
sleep $INTERVAL
T_START=$(date +%s)
END=$(( T_START + DURATION ))
echo ""
echo "=== Olcum basladi ==="
while [ $(date +%s) -lt $END ]; do
    TS=$(date +%s%3N)
    ELAPSED=$(( $(date +%s) - T_START ))
    if   [ $ELAPSED -lt 30 ]; then PHASE="normal"
    elif [ $ELAPSED -lt 45 ]; then PHASE="burst"
    elif [ $ELAPSED -lt 60 ]; then PHASE="idle"
    else                            PHASE="burst2"
    fi
    read UE_RX2 UE_TX2 <<< $(read_bytes $UE_IFACE)
    read OG_RX2 OG_TX2 <<< $(read_bytes ogstun)
    read E4_RX2 E4_TX2 <<< $(read_bytes ens4)
    DT_MS=$(( TS - T_PREV_MS ))
    if [ -z "$UE_TX2" ] || [ -z "$UE_RX2" ]; then
        UE_IFACE=$(get_ue_iface)
        [ -z "$UE_IFACE" ] && echo "  UYARI: UE tunnel kayboldu!" \
            && sleep $INTERVAL && continue
        read UE_RX2 UE_TX2 <<< $(read_bytes $UE_IFACE)
    fi
    UE_TX_K=$(calc_kbps ${UE_TX1:-0} ${UE_TX2:-0} $DT_MS)
    UE_RX_K=$(calc_kbps ${UE_RX1:-0} ${UE_RX2:-0} $DT_MS)
    OG_TX_K=$(calc_kbps ${OG_TX1:-0} ${OG_TX2:-0} $DT_MS)
    OG_RX_K=$(calc_kbps ${OG_RX1:-0} ${OG_RX2:-0} $DT_MS)
    E4_TX_K=$(calc_kbps ${E4_TX1:-0} ${E4_TX2:-0} $DT_MS)
    E4_RX_K=$(calc_kbps ${E4_RX1:-0} ${E4_RX2:-0} $DT_MS)
    SESS=$(curl -s http://127.0.0.7:9090/metrics 2>/dev/null | \
        grep 'upf_sessionnbr' | grep -v '#' | awk '{print $2}' || echo 0)
    echo "${TS},${PHASE},${UE_TX_K},${UE_RX_K},${OG_TX_K},${OG_RX_K},${E4_TX_K},${E4_RX_K},${SESS:-0}" >> $LOG
    printf "  [%2ds|%-7s] UE_TX=%-10s UE_RX=%-10s KB/s  (dt=%sms)\n" \
        "$ELAPSED" "$PHASE" "$UE_TX_K" "$UE_RX_K" "$DT_MS"
    UE_RX1=$UE_RX2; UE_TX1=$UE_TX2
    OG_RX1=$OG_RX2; OG_TX1=$OG_TX2
    E4_RX1=$E4_RX2; E4_TX1=$E4_TX2
    T_PREV_MS=$TS
    sleep $INTERVAL
done
wait $TRAFFIC_PID 2>/dev/null
echo ""
echo "=== TAMAMLANDI: $(( $(wc -l < $LOG) - 1 )) adim ==="
echo "Max UE_TX KB/s:"
tail -n +2 $LOG | awk -F',' '{if($3+0>max) max=$3+0} END{printf "  %.2f\n",max}'
echo "Max UE_RX KB/s:"
tail -n +2 $LOG | awk -F',' '{if($4+0>max) max=$4+0} END{printf "  %.2f\n",max}'
echo "Sifir olmayan adim:"
tail -n +2 $LOG | awk -F',' '{if($3+0>0||$4+0>0) n++} END{print "  "n+0}'
SCRIPT
gcloud compute scp /tmp/scenario_c.sh $VM_NAME:/tmp/scenario_c.sh \
    --project=$PROJECT_ID --zone=$ZONE
gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE \
    --command="sudo bash /tmp/scenario_c.sh"
gcloud compute scp $VM_NAME:/tmp/scenario_c.csv ./scenario_c.csv \
    --project=$PROJECT_ID --zone=$ZONE
echo "✅ scenario_c.csv: $(( $(wc -l < ./scenario_c.csv) - 1 )) adim"

In [ ]:









# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 6 — MULTI-UE PROFİL SİMÜLASYONU                                    ║
# ╠══════════════════════════════════════════════════════════════════════════════╣
# ║  Açıklama:                                                                  ║
# ║    Tek UE üzerinden 3 farklı cihaz profili simüle eder.                    ║
# ║    UE1=mobile_broadband / UE2=iot_sensor / UE3=video_stream               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# %%bash
# PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"
#
# cat > /tmp/scenario_multi_ue.sh << 'SCRIPT'
# #!/bin/bash
# LOG="/tmp/scenario_multi_ue.csv"
# NR="/opt/UERANSIM/build"
# UE="uesimtun0"
# UE_IP=$(ip a show $UE | grep 'inet ' | awk '{print $2}' | cut -d'/' -f1)
# echo "timestamp_ms,ue_id,ue_ip,iface,target,protocol,bytes,rtt_ms,profile,status" > $LOG
# log() { echo "$(date +%s%3N),$1,$2,$3,$4,$5,$6,$7,$8,$9" >> $LOG; }
#
# echo "[UE1] Mobile Broadband"
# for i in $(seq 1 10); do
#     cd $NR
#     RESULT=$(./nr-binder $UE curl -s -o /dev/null \
#         -w "%{size_download}|%{time_total}" \
#         --max-time 15 "https://www.google.com" 2>/dev/null || echo "0|0")
#     SIZE=$(echo $RESULT | cut -d'|' -f1)
#     TIME=$(echo $RESULT | cut -d'|' -f2 | awk '{printf "%.0f", $1*1000}')
#     log "UE1" "$UE_IP" "$UE" "www.google.com" "HTTP" "${SIZE:-0}" "${TIME:-0}" "mobile_broadband" "OK"
#     sleep 2
# done
# for i in $(seq 1 5); do
#     cd $NR
#     RESULT=$(./nr-binder $UE curl -s -o /dev/null \
#         -w "%{size_download}|%{time_total}" \
#         --max-time 15 "https://www.cloudflare.com" 2>/dev/null || echo "0|0")
#     SIZE=$(echo $RESULT | cut -d'|' -f1)
#     TIME=$(echo $RESULT | cut -d'|' -f2 | awk '{printf "%.0f", $1*1000}')
#     log "UE1" "$UE_IP" "$UE" "www.cloudflare.com" "HTTP" "${SIZE:-0}" "${TIME:-0}" "mobile_broadband" "OK"
#     sleep 2
# done
#
# echo "[UE2] IoT Sensor"
# for i in $(seq 1 30); do
#     cd $NR
#     RTT=$(./nr-binder $UE ping -c 1 -s 16 -q 8.8.8.8 2>&1 | \
#         grep rtt | awk -F'/' '{print $5}')
#     log "UE2" "$UE_IP" "$UE" "8.8.8.8" "ICMP" "16" "${RTT:-0}" "iot_sensor" "OK"
#     sleep 0.5
# done
# for i in $(seq 1 10); do
#     cd $NR
#     RTT=$(./nr-binder $UE ping -c 1 -s 16 -q 1.1.1.1 2>&1 | \
#         grep rtt | awk -F'/' '{print $5}')
#     log "UE2" "$UE_IP" "$UE" "1.1.1.1" "ICMP" "16" "${RTT:-0}" "iot_sensor" "OK"
#     sleep 0.5
# done
#
# echo "[UE3] Video Streaming"
# for i in $(seq 1 10); do
#     cd $NR
#     RESULT=$(./nr-binder $UE curl -s -o /dev/null \
#         -w "%{size_download}|%{time_total}" \
#         --max-time 20 "https://www.cloudflare.com" 2>/dev/null || echo "0|0")
#     SIZE=$(echo $RESULT | cut -d'|' -f1)
#     TIME=$(echo $RESULT | cut -d'|' -f2 | awk '{printf "%.0f", $1*1000}')
#     log "UE3" "$UE_IP" "$UE" "www.cloudflare.com" "HTTP" "${SIZE:-0}" "${TIME:-0}" "video_stream" "OK"
#     sleep 3
# done
# for i in $(seq 1 10); do
#     cd $NR
#     RTT=$(./nr-binder $UE ping -c 1 -s 1400 -q 8.8.8.8 2>&1 | \
#         grep rtt | awk -F'/' '{print $5}')
#     log "UE3" "$UE_IP" "$UE" "8.8.8.8" "ICMP" "1400" "${RTT:-0}" "video_stream" "OK"
#     sleep 1
# done
#
# echo "=== TAMAMLANDI: $(( $(wc -l < $LOG) - 1 )) kayit ==="
# SCRIPT
#
# gcloud compute scp /tmp/scenario_multi_ue.sh $VM_NAME:/tmp/scenario_multi_ue.sh \
#     --project=$PROJECT_ID --zone=$ZONE
# gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE \
#     --command="sudo bash /tmp/scenario_multi_ue.sh"
# gcloud compute scp $VM_NAME:/tmp/scenario_multi_ue.csv ./scenario_multi_ue.csv \
#     --project=$PROJECT_ID --zone=$ZONE
# echo "✅ Multi-UE dataset: $(( $(wc -l < ./scenario_multi_ue.csv) - 1 )) kayit"


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 7 — 5G-A: HANDOVER SİMÜLASYONU  [v2 - DÜZELTİLDİ]                ║
# ╠══════════════════════════════════════════════════════════════════════════════╣
# ║  Açıklama:                                                                  ║
# ║    UE detach → re-attach döngüsünü ölçer.                                  ║
# ║    v2 Düzeltmeleri:                                                         ║
# ║    • get_ue_iface(): HO sonrası değişen interface'i dinamik tespit eder    ║
# ║    • measure_rtt() hardcoded uesimtun0 → parametre ile dinamik             ║
# ║    • "514msms" typo düzeltildi (echo "HO_DUR ms")                         ║
# ║    • Tunnel kaybolana kadar bekleme + re-attach timeout 30sn               ║
# ║    Fazlar: PRE_HANDOVER → HANDOVER → HO_GAP → POST_HANDOVER → STEADY_STATE║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# %%bash
# PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"
#
# cat > /tmp/scenario_handover.sh << 'SCRIPT'
# #!/bin/bash
# LOG="/tmp/scenario_handover.csv"
# NR="/opt/UERANSIM/build"
# echo "timestamp_ms,phase,event,ue_ip,target,rtt_ms,status,note" > $LOG
#
# log() { echo "$(date +%s%3N),$1,$2,$3,$4,$5,$6,$7" >> $LOG; }
#
# get_ue_iface() {
#     for iface in uesimtun0 uesimtun1 uesimtun2 uesimtun3; do
#         IP=$(ip a show $iface 2>/dev/null | grep 'inet ' | \
#             awk '{print $2}' | cut -d'/' -f1)
#         [ -n "$IP" ] && echo "$iface $IP" && return
#     done
#     echo ""
# }
#
# measure_rtt() {
#     local IFACE=$1
#     [ -z "$IFACE" ] && echo "0" && return
#     RTT=$(cd $NR && ./nr-binder $IFACE ping -c 1 -q 8.8.8.8 2>&1 | \
#         grep rtt | awk -F'/' '{print $5}')
#     echo "${RTT:-0}"
# }
#
# read IFACE UE_IP <<< $(get_ue_iface)
# if [ -z "$IFACE" ]; then echo "HATA: Aktif UE tunnel bulunamadi"; exit 1; fi
# echo "Aktif UE: $IFACE ($UE_IP)"
# echo "=== HANDOVER SENARYOSU ==="
#
# echo "[HO] Faz 1: Pre-handover baseline (10 olcum)"
# for i in $(seq 1 10); do
#     RTT=$(measure_rtt $IFACE)
#     log "PRE_HANDOVER" "BASELINE" "$UE_IP" "8.8.8.8" "$RTT" "OK" "normal_ops"
#     sleep 0.5
# done
# PRE_RTT_AVG=$(grep PRE_HANDOVER $LOG | \
#     awk -F',' '{sum+=$6; n++} END{printf "%.3f", sum/n}')
# echo "  Pre-HO RTT ort: ${PRE_RTT_AVG}ms"
#
# echo "[HO] Faz 2: Handover baslıyor (UE detach)"
# log "HANDOVER" "HO_START" "$UE_IP" "-" "0" "DISCONN" "ue_detach"
# sudo pkill -f nr-ue || true
# sleep 2
# for i in $(seq 1 10); do
#     sleep 0.5
#     STILL=$(ip a show $IFACE 2>/dev/null | grep 'inet ')
#     [ -z "$STILL" ] && break
# done
# for i in $(seq 1 5); do
#     log "HANDOVER" "HO_GAP" "-" "8.8.8.8" "0" "NO_SESSION" "tunnel_down"
#     sleep 0.3
# done
#
# echo "[HO] Faz 3: Re-attachment"
# HO_TS=$(date +%s%3N)
# sudo nohup $NR/nr-ue \
#     -c /opt/UERANSIM/config/open5gs-ue.yaml > /root/ue.log 2>&1 &
# NEW_IFACE=""; NEW_IP=""
# for i in $(seq 1 60); do
#     sleep 0.5
#     read NEW_IFACE NEW_IP <<< $(get_ue_iface)
#     if [ -n "$NEW_IFACE" ] && [ -n "$NEW_IP" ]; then
#         HO_DUR=$(( $(date +%s%3N) - HO_TS ))
#         log "HANDOVER" "HO_COMPLETE" "$NEW_IP" "-" "0" "RECONNECTED" \
#             "ho_duration=${HO_DUR}ms"
#         echo "  HO tamamlandi: ${HO_DUR}ms | Yeni: $NEW_IFACE ($NEW_IP)"
#         break
#     fi
# done
# if [ -z "$NEW_IFACE" ]; then echo "HATA: Re-attachment basarisiz"; exit 1; fi
# sleep 3
#
# echo "[HO] Faz 4: Post-handover RTT spike (15 olcum)"
# for i in $(seq 1 15); do
#     RTT=$(measure_rtt $NEW_IFACE)
#     CUR_IP=$(ip a show $NEW_IFACE 2>/dev/null | \
#         grep 'inet ' | awk '{print $2}' | cut -d'/' -f1)
#     STATUS=$([ "${RTT:-0}" != "0" ] && echo "OK" || echo "FAIL")
#     log "POST_HANDOVER" "RECOVERY" "${CUR_IP:-$NEW_IP}" \
#         "8.8.8.8" "${RTT:-0}" "$STATUS" "post_ho_${i}"
#     sleep 0.5
# done
#
# echo "[HO] Faz 5: Steady state (10 olcum)"
# for i in $(seq 1 10); do
#     RTT=$(measure_rtt $NEW_IFACE)
#     CUR_IP=$(ip a show $NEW_IFACE 2>/dev/null | \
#         grep 'inet ' | awk '{print $2}' | cut -d'/' -f1)
#     log "STEADY_STATE" "STABLE" "${CUR_IP:-$NEW_IP}" \
#         "8.8.8.8" "${RTT:-0}" "OK" "stabilized"
#     sleep 1
# done
#
# echo ""
# echo "=== HANDOVER ANALIZI ==="
# echo "Pre-HO RTT ort:"
# grep PRE_HANDOVER $LOG | awk -F',' '{sum+=$6;n++} END{printf "  %.3f ms\n",sum/n}'
# echo "Post-HO RTT ort (ilk 5):"
# grep POST_HANDOVER $LOG | head -5 | awk -F',' '{sum+=$6;n++} END{printf "  %.3f ms\n",sum/n}'
# echo "Steady RTT ort:"
# grep STEADY_STATE $LOG | awk -F',' '{sum+=$6;n++} END{printf "  %.3f ms\n",sum/n}'
# echo "HO suresi:"
# grep HO_COMPLETE $LOG | awk -F'=' '{print "  "$NF}'
# echo "Toplam kayit: $(( $(wc -l < $LOG) - 1 ))"
# SCRIPT
#
# gcloud compute scp /tmp/scenario_handover.sh $VM_NAME:/tmp/scenario_handover.sh \
#     --project=$PROJECT_ID --zone=$ZONE
# gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE \
#     --command="sudo bash /tmp/scenario_handover.sh"
# gcloud compute scp $VM_NAME:/tmp/scenario_handover.csv ./scenario_handover.csv \
#     --project=$PROJECT_ID --zone=$ZONE
# echo "✅ Handover dataset: $(( $(wc -l < ./scenario_handover.csv) - 1 )) kayit"


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 8 — 5G-B: PDU SESSION TEARDOWN & RE-ESTABLISH  [v2 - DÜZELTİLDİ]  ║
# ╠══════════════════════════════════════════════════════════════════════════════╣
# ║  Açıklama:                                                                  ║
# ║    3 cycle PDU session açma/kapama ölçümü.                                 ║
# ║    v2 Düzeltmeleri:                                                         ║
# ║    • Başlangıç temizliği: UPF_SESS=0 olana kadar SMF/UPF restart fallback ║
# ║    • nr-cli ile düzgün 5G deregistration (teardown 22sn → ~2sn hedef)     ║
# ║    • get_ue_iface() dinamik interface takibi                                ║
# ║    • Teardown bekleme: pkill yerine NAS deregistration prosedürü           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# %%bash
# PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"
#
# cat > /tmp/scenario_session.sh << 'SCRIPT'
# #!/bin/bash
# LOG="/tmp/scenario_session.csv"
# NR="/opt/UERANSIM/build"
# echo "timestamp_ms,event,session_id,ue_ip,rtt_ms,upf_sessions,duration_ms,status" > $LOG
#
# log() { echo "$(date +%s%3N),$1,$2,$3,$4,$5,$6,$7" >> $LOG; }
#
# get_upf_sess() {
#     curl -s http://127.0.0.7:9090/metrics 2>/dev/null | \
#         grep 'upf_sessionnbr' | grep -v '#' | awk '{print $2}' || echo 0
# }
#
# get_ue_iface() {
#     for iface in uesimtun0 uesimtun1 uesimtun2 uesimtun3; do
#         IP=$(ip a show $iface 2>/dev/null | grep 'inet ' | \
#             awk '{print $2}' | cut -d'/' -f1)
#         [ -n "$IP" ] && echo "$iface $IP" && return
#     done
#     echo ""
# }
#
# measure_rtt() {
#     local IFACE=$1
#     cd $NR && ./nr-binder $IFACE ping -c 1 -q 8.8.8.8 2>&1 | \
#         grep rtt | awk -F'/' '{print $5}'
# }
#
# echo "=== BASLANGIC TEMIZLIGI ==="
# sudo pkill -f nr-ue || true
# sleep 2
# SESS=$(get_upf_sess)
# if [ "$SESS" != "0" ]; then
#     echo "  SMF/UPF restart (UPF_SESS=$SESS)..."
#     sudo systemctl restart open5gs-smfd open5gs-upfd
#     sleep 5
#     echo "  Restart sonrasi: UPF_SESS=$(get_upf_sess)"
# fi
#
# echo "  UE baslatiliyor..."
# sudo nohup $NR/nr-ue \
#     -c /opt/UERANSIM/config/open5gs-ue.yaml > /root/ue.log 2>&1 &
# for i in $(seq 1 40); do
#     sleep 0.5
#     read IFACE UE_IP <<< $(get_ue_iface)
#     [ -n "$UE_IP" ] && break
# done
# echo "  Hazir: $IFACE ($UE_IP) | UPF_SESS=$(get_upf_sess)"
#
# echo ""
# echo "=== SESSION MANAGEMENT SENARYOSU ==="
#
# for SESSION_ID in $(seq 1 3); do
#     echo "─── Session Cycle $SESSION_ID / 3 ───"
#     read IFACE UE_IP <<< $(get_ue_iface)
#     SESS=$(get_upf_sess)
#     RTT=$(measure_rtt "$IFACE")
#     log "SESSION_ACTIVE" "$SESSION_ID" "$UE_IP" "${RTT:-0}" "$SESS" "0" "OK"
#     echo "  [Aktif] $IFACE ($UE_IP) RTT=${RTT}ms UPF_SESS=$SESS"
#
#     for i in $(seq 1 5); do
#         RTT=$(measure_rtt "$IFACE")
#         log "DATA_TRANSFER" "$SESSION_ID" "$UE_IP" "${RTT:-0}" "$(get_upf_sess)" "0" "OK"
#         sleep 1
#     done
#
#     TEAR_TS=$(date +%s%3N)
#     log "SESSION_TEARDOWN" "$SESSION_ID" "$UE_IP" "0" "$(get_upf_sess)" "0" "TEARING_DOWN"
#     echo "  [Teardown] nr-cli ile deregister..."
#
#     UE_NODE=$(cd $NR && ./nr-cli --dump 2>/dev/null | grep 'imsi-' | head -1 | awk '{print $1}')
#     if [ -n "$UE_NODE" ]; then
#         cd $NR && echo "deregister disable-5g" | ./nr-cli "$UE_NODE" 2>/dev/null
#         sleep 2
#         for i in $(seq 1 20); do
#             SESS=$(get_upf_sess)
#             ELAPSED=$(( $(date +%s%3N) - TEAR_TS ))
#             log "SESSION_RELEASING" "$SESSION_ID" "-" "0" "$SESS" "$ELAPSED" "RELEASING"
#             [ "$SESS" = "0" ] && echo "  UPF session sifir: ${ELAPSED}ms" && break
#             sleep 0.5
#         done
#     else
#         echo "  [Fallback] pkill + SMF restart..."
#         sudo pkill -f nr-ue || true
#         sleep 2
#         sudo systemctl restart open5gs-smfd
#         sleep 4
#         log "SESSION_RELEASING" "$SESSION_ID" "-" "0" "0" \
#             "$(( $(date +%s%3N) - TEAR_TS ))" "RESTARTED"
#     fi
#
#     TEAR_DUR=$(( $(date +%s%3N) - TEAR_TS ))
#     log "SESSION_RELEASED" "$SESSION_ID" "-" "0" "0" "$TEAR_DUR" "RELEASED"
#     echo "  [Released] ${TEAR_DUR}ms"
#
#     sudo pkill -f nr-ue || true
#     sleep 2
#
#     ESTAB_TS=$(date +%s%3N)
#     echo "  [Re-establish] Baslatiliyor..."
#     sudo nohup $NR/nr-ue \
#         -c /opt/UERANSIM/config/open5gs-ue.yaml > /root/ue.log 2>&1 &
#
#     for i in $(seq 1 60); do
#         sleep 0.5
#         read NEW_IFACE NEW_IP <<< $(get_ue_iface)
#         SESS=$(get_upf_sess)
#         ELAPSED=$(( $(date +%s%3N) - ESTAB_TS ))
#         log "SESSION_ESTABLISHING" "$SESSION_ID" "${NEW_IP:--}" "0" "$SESS" "$ELAPSED" "WAIT"
#         if [ -n "$NEW_IP" ] && [ "$SESS" != "0" ]; then
#             log "SESSION_ESTABLISHED" "$SESSION_ID" "$NEW_IP" "0" "$SESS" "$ELAPSED" "OK"
#             echo "  [Established] IP=$NEW_IP | ${ELAPSED}ms | UPF=$SESS"
#             IFACE=$NEW_IFACE; UE_IP=$NEW_IP
#             break
#         fi
#     done
#     sleep 3
# done
#
# echo ""
# echo "=== SESSION OZETI ==="
# echo "Teardown (ort):"
# grep SESSION_RELEASED $LOG | \
#     awk -F',' '{sum+=$7;n++} END{printf "  %.0f ms\n",sum/n}'
# echo "Re-establishment (ort):"
# grep 'SESSION_ESTABLISHED$' $LOG | \
#     awk -F',' '{sum+=$7;n++} END{printf "  %.0f ms\n",sum/n}'
# echo "Toplam kayit: $(( $(wc -l < $LOG) - 1 ))"
# SCRIPT
#
# gcloud compute scp /tmp/scenario_session.sh $VM_NAME:/tmp/scenario_session.sh \
#     --project=$PROJECT_ID --zone=$ZONE
# gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE \
#     --command="sudo bash /tmp/scenario_session.sh"
# gcloud compute scp $VM_NAME:/tmp/scenario_session.csv ./scenario_session.csv \
#     --project=$PROJECT_ID --zone=$ZONE
# echo "✅ Session dataset: $(( $(wc -l < ./scenario_session.csv) - 1 )) kayit"


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 9 — 5G-C: QoS FARKLILASTIRMA (eMBB / mMTC / URLLC)                ║
# ╠══════════════════════════════════════════════════════════════════════════════╣
# ║  Açıklama:                                                                  ║
# ║    3GPP 5QI sınıflarını trafik davranışıyla simüle eder.                   ║
# ║    URLLC SLA = 5ms hedef, SLA_MET/SLA_BREACH etiketlenir.                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# %%bash
# PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"
#
# cat > /tmp/scenario_qos.sh << 'SCRIPT'
# #!/bin/bash
# LOG="/tmp/scenario_qos.csv"
# NR="/opt/UERANSIM/build"
# echo "timestamp_ms,dnn,ue_ip,iface,target,protocol,bytes,rtt_ms,qos_class,status" > $LOG
# log() { echo "$(date +%s%3N),$1,$2,$3,$4,$5,$6,$7,$8,$9" >> $LOG; }
# UE_IP=$(ip a show uesimtun0 | grep 'inet ' | awk '{print $2}' | cut -d'/' -f1)
#
# echo "[QoS] eMBB — buyuk HTTP"
# for i in $(seq 1 15); do
#     cd $NR
#     RESULT=$(./nr-binder uesimtun0 curl -s -o /dev/null \
#         -w "%{size_download}|%{time_total}" \
#         --max-time 10 "https://www.google.com" 2>/dev/null || echo "0|0")
#     SIZE=$(echo $RESULT | cut -d'|' -f1)
#     TIME=$(echo $RESULT | cut -d'|' -f2 | awk '{printf "%.0f", $1*1000}')
#     log "internet" "$UE_IP" "uesimtun0" "www.google.com" "HTTP" \
#         "${SIZE:-0}" "${TIME:-0}" "eMBB" "OK"
#     sleep 1
# done
#
# echo "[QoS] mMTC — IoT"
# for i in $(seq 1 20); do
#     cd $NR
#     RTT=$(./nr-binder uesimtun0 ping -c 1 -q 8.8.8.8 2>&1 | \
#         grep rtt | awk -F'/' '{print $5}')
#     log "internet" "$UE_IP" "uesimtun0" "8.8.8.8" "ICMP" \
#         "84" "${RTT:-0}" "mMTC" "OK"
#     sleep 0.5
# done
#
# echo "[QoS] URLLC — ultra-dusuk gecikme"
# for i in $(seq 1 30); do
#     cd $NR
#     RTT=$(./nr-binder uesimtun0 ping -c 1 -q 1.1.1.1 2>&1 | \
#         grep rtt | awk -F'/' '{print $5}')
#     SLA=$(echo "${RTT:-999}" | awk '{print ($1 < 5) ? "SLA_MET" : "SLA_BREACH"}')
#     log "internet" "$UE_IP" "uesimtun0" "1.1.1.1" "ICMP" \
#         "84" "${RTT:-0}" "URLLC" "$SLA"
#     sleep 0.3
# done
#
# echo "=== QoS OZETI ==="
# echo "eMBB ort byte:"; grep eMBB $LOG | awk -F',' '{sum+=$7;n++} END{printf "  %.0f byte\n",sum/n}'
# echo "mMTC ort RTT:"; grep mMTC $LOG | awk -F',' '{sum+=$8;n++} END{printf "  %.3f ms\n",sum/n}'
# echo "URLLC SLA:"; grep URLLC $LOG | \
#     awk -F',' '{n++; if($10=="SLA_MET") m++} END{printf "  %.1f%%\n",m/n*100}'
# SCRIPT
#
# gcloud compute scp /tmp/scenario_qos.sh $VM_NAME:/tmp/scenario_qos.sh \
#     --project=$PROJECT_ID --zone=$ZONE
# gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE \
#     --command="sudo bash /tmp/scenario_qos.sh"
# gcloud compute scp $VM_NAME:/tmp/scenario_qos.csv ./scenario_qos.csv \
#     --project=$PROJECT_ID --zone=$ZONE
# echo "✅ QoS dataset: $(wc -l < ./scenario_qos.csv) satir"


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 10 — 5G-D: NETWORK SLICING (eMBB / URLLC / mMTC)                  ║
# ╠══════════════════════════════════════════════════════════════════════════════╣
# ║  Açıklama:                                                                  ║
# ║    SST=1/2/3 etiketli 3 farklı slice trafik profili üretir.               ║
# ║    URLLC SLA_MET/SLA_BREACH etiketi, eMBB throughput KB/s hesabı dahil.  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# %%bash
# PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"
#
# cat > /tmp/scenario_slice.sh << 'SCRIPT'
# #!/bin/bash
# LOG="/tmp/scenario_slice.csv"
# NR="/opt/UERANSIM/build"
# echo "timestamp_ms,slice_type,sst,sd,ue_ip,target,protocol,bytes,rtt_ms,slice_kpi,status" > $LOG
# log() { echo "$(date +%s%3N),$1,$2,$3,$4,$5,$6,$7,$8,$9,$10" >> $LOG; }
# UE_IP=$(ip a show uesimtun0 | grep 'inet ' | awk '{print $2}' | cut -d'/' -f1)
#
# echo "[SLICE] eMBB (SST=1)"
# for i in $(seq 1 10); do
#     cd $NR
#     RESULT=$(./nr-binder uesimtun0 curl -s -o /dev/null \
#         -w "%{size_download}|%{time_total}" \
#         --max-time 15 "https://www.cloudflare.com" 2>/dev/null || echo "0|0")
#     SIZE=$(echo $RESULT | cut -d'|' -f1)
#     TIME=$(echo $RESULT | cut -d'|' -f2 | awk '{printf "%.0f", $1*1000}')
#     TPUT=$(echo "scale=2; ${SIZE:-0} / (${TIME:-1} / 1000) / 1024" | bc 2>/dev/null || echo 0)
#     log "eMBB" "1" "0x000001" "$UE_IP" "cloudflare.com" \
#         "HTTP" "${SIZE:-0}" "${TIME:-0}" "${TPUT}KB/s" "OK"
#     sleep 2
# done
#
# echo "[SLICE] URLLC (SST=2)"
# for i in $(seq 1 30); do
#     cd $NR
#     RTT=$(./nr-binder uesimtun0 ping -c 1 -q 1.1.1.1 2>&1 | \
#         grep rtt | awk -F'/' '{print $5}')
#     SLA=$(echo "${RTT:-999}" | awk '{print ($1 < 5) ? "SLA_MET" : "SLA_BREACH"}')
#     log "URLLC" "2" "0x000002" "$UE_IP" "1.1.1.1" \
#         "ICMP" "84" "${RTT:-0}" "$SLA" "OK"
#     sleep 0.3
# done
#
# echo "[SLICE] mMTC (SST=3)"
# for i in $(seq 1 40); do
#     cd $NR
#     RTT=$(./nr-binder uesimtun0 ping -c 1 -s 16 -q 8.8.8.8 2>&1 | \
#         grep rtt | awk -F'/' '{print $5}')
#     log "mMTC" "3" "0x000003" "$UE_IP" "8.8.8.8" \
#         "ICMP" "16" "${RTT:-0}" "IoT_HEARTBEAT" "OK"
#     sleep 2
# done
#
# echo "=== SLICE KPI ==="
# echo "eMBB ort size:"; grep eMBB $LOG | awk -F',' '{sum+=$8;n++} END{printf "  %.0f byte\n",sum/n}'
# echo "URLLC SLA:"; grep URLLC $LOG | \
#     awk -F',' '{n++; if($10=="SLA_MET") m++} END{printf "  %.1f%% met\n",m/n*100}'
# echo "mMTC mesaj:"; grep mMTC $LOG | wc -l | awk '{printf "  %d\n",$1}'
# SCRIPT
#
# gcloud compute scp /tmp/scenario_slice.sh $VM_NAME:/tmp/scenario_slice.sh \
#     --project=$PROJECT_ID --zone=$ZONE
# gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE \
#     --command="sudo bash /tmp/scenario_slice.sh"
# gcloud compute scp $VM_NAME:/tmp/scenario_slice.csv ./scenario_slice.csv \
#     --project=$PROJECT_ID --zone=$ZONE
# echo "✅ Slice dataset: $(wc -l < ./scenario_slice.csv) satir"


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 11 — TÜM VERİYİ COLAB'A İNDİR                                      ║
# ╠══════════════════════════════════════════════════════════════════════════════╣
# ║  Açıklama: VM'deki tüm CSV'leri tek seferde Colab'a indirir.              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# %%bash
# PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"
#
# FILES="scenario_a.csv scenario_b.csv scenario_c.csv scenario_multi_ue.csv
#        scenario_handover.csv scenario_session.csv scenario_qos.csv scenario_slice.csv"
#
# for FILE in $FILES; do
#     gcloud compute scp $VM_NAME:/tmp/$FILE ./$FILE \
#         --project=$PROJECT_ID --zone=$ZONE 2>/dev/null \
#         && echo "✅ $FILE — $(( $(wc -l < ./$FILE) - 1 )) kayit" \
#         || echo "⚠️  $FILE bulunamadi"
# done


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 12 — ETL: PANDAS DATAFRAME'LERE YÜKLE                              ║
# ╠══════════════════════════════════════════════════════════════════════════════╣
# ║  Açıklama:                                                                  ║
# ║    Tüm CSV'leri yükler, timestamp dönüşümü ve sayısal cast yapar.          ║
# ║    dfs{} sözlüğünde saklanır.                                              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import pandas as pd
import numpy as np
import os

dfs = {}

loaders = {
    'A':        'scenario_a.csv',
    'B':        'scenario_b.csv',
    'C':        'scenario_c.csv',
    'multi_ue': 'scenario_multi_ue.csv',
    'handover': 'scenario_handover.csv',
    'session':  'scenario_session.csv',
    'qos':      'scenario_qos.csv',
    'slice':    'scenario_slice.csv',
}

for key, fname in loaders.items():
    if os.path.exists(fname):
        df = pd.read_csv(fname)
        df['timestamp'] = pd.to_datetime(df['timestamp_ms'], unit='ms')
        for col in df.columns:
            if col not in ['timestamp', 'timestamp_ms']:
                converted = pd.to_numeric(df[col], errors='coerce')
                if converted.notna().mean() > 0.2:
                    df[col] = converted
        dfs[key] = df
        print(f"✅ {fname}: {len(df)} kayit")
    else:
        print(f"⚠️  {fname}: bulunamadi")

print(f"\n✅ Toplam {len(dfs)} dataset: {list(dfs.keys())}")


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 13 — FEATURE ENGINEERING                                             ║
# ╠══════════════════════════════════════════════════════════════════════════════╣
# ║  Türetilen özellikler:                                                       ║
# ║  • inter_arrival_ms : Ardışık paket arası süre (anomali sinyali)           ║
# ║  • bytes_per_ms     : Anlık throughput göstergesi                          ║
# ║  • is_icmp/http/dns : Protokol binary flag                                 ║
# ║  • rtt_zscore       : |zscore|>3 → istatistiksel anomali                  ║
# ║  • is_anomaly       : Senaryo B için binary label                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

for key in ['A', 'B', 'multi_ue']:
    if key not in dfs:
        continue
    df = dfs[key]
    df['rtt_ms']     = pd.to_numeric(df['rtt_ms'], errors='coerce').fillna(0)
    df['bytes']      = pd.to_numeric(df['bytes'],  errors='coerce').fillna(0)
    df['inter_arrival_ms'] = df['timestamp'].diff().dt.total_seconds().mul(1000).fillna(0)
    df['bytes_per_ms']     = np.where(df['rtt_ms'] > 0, df['bytes'] / df['rtt_ms'], 0)
    df['is_dns']           = (df['protocol'] == 'DNS').astype(int)
    df['is_icmp']          = (df['protocol'] == 'ICMP').astype(int)
    df['is_http']          = (df['protocol'] == 'HTTP').astype(int)
    rtt_std = df['rtt_ms'].std()
    df['rtt_zscore']       = (df['rtt_ms'] - df['rtt_ms'].mean()) / (rtt_std + 1e-9)
    if 'label' in df.columns:
        df['is_anomaly'] = df['label'].str.startswith('ANOMALY').astype(int)
    print(f"✅ Senaryo {key} feature engineering tamamlandi")
    print(df[['rtt_ms','bytes','inter_arrival_ms',
              'bytes_per_ms','rtt_zscore']].describe().round(3))
    print()


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 14 — GÖRSELLEŞTİRME: SENARYO A / B / C                            ║
# ╠══════════════════════════════════════════════════════════════════════════════╣
# ║  Plot 1: RTT zaman serisi (protokol renkli)                                 ║
# ║  Plot 2: Protokol byte dağılımı (bar)                                       ║
# ║  Plot 3: Inter-arrival histogram                                             ║
# ║  Plot 4: Normal vs Anomali scatter                                           ║
# ║  Plot 5: Senaryo C throughput (KB/s)                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sns.set_theme(style="darkgrid")

COLORS_A = {'A_ICMP':'#2196F3','A_HTTP':'#00BCD4','A_DNS':'#8BC34A','A_BURST':'#FF5722'}
COLORS_B = {'NORMAL':'#4CAF50','ANOMALY_SCAN':'#FF9800','ANOMALY_FLOOD':'#F44336',
            'ANOMALY_SLOWLORIS':'#9C27B0','ANOMALY_EXFIL':'#795548'}

fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle('5G AI/ML Lab — Temel Senaryo Analizi (A/B/C)',
             fontsize=14, fontweight='bold')

if 'A' in dfs:
    ax = fig.add_subplot(gs[0, :])
    df_rtt = dfs['A'][dfs['A']['rtt_ms'] > 0]
    for sc, grp in df_rtt.groupby('scenario'):
        ax.scatter(grp['timestamp'], grp['rtt_ms'],
                   label=sc, c=COLORS_A.get(sc,'gray'), s=55, alpha=0.85, zorder=3)
    ax.axhline(df_rtt['rtt_ms'].mean(), color='red', linestyle='--', linewidth=1.2,
               label=f"Ort: {df_rtt['rtt_ms'].mean():.2f}ms")
    ax.set_title('Senaryo A — RTT Zaman Serisi')
    ax.set_ylabel('RTT (ms)')
    ax.legend(ncol=5, fontsize=9)
    ax.tick_params(axis='x', rotation=20)

if 'A' in dfs:
    ax = fig.add_subplot(gs[1, 0])
    proto_bytes = dfs['A'].groupby('protocol')['bytes'].sum() / 1024
    proto_bytes.plot(kind='bar', ax=ax,
                     color=['#2196F3','#4CAF50','#FF9800'],
                     edgecolor='white', alpha=0.85)
    ax.set_title('Senaryo A — Protokol Byte Dağılımı')
    ax.set_ylabel('Toplam (KB)')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)

if 'A' in dfs:
    ax = fig.add_subplot(gs[1, 1])
    dfs['A'][dfs['A']['inter_arrival_ms'] > 0]['inter_arrival_ms'] \
        .clip(upper=10000) \
        .hist(ax=ax, bins=30, color='#2196F3', edgecolor='white', alpha=0.85)
    ax.set_title('Senaryo A — Inter-Arrival Time')
    ax.set_xlabel('ms')
    ax.set_ylabel('Frekans')

if 'B' in dfs:
    ax = fig.add_subplot(gs[2, 0])
    for label, grp in dfs['B'].groupby('label'):
        ax.scatter(grp['inter_arrival_ms'].clip(upper=20000),
                   grp['rtt_ms'],
                   label=label, c=COLORS_B.get(label,'gray'), s=55, alpha=0.75)
    ax.set_title('Senaryo B — Normal vs Anomali')
    ax.set_xlabel('Inter-Arrival (ms)')
    ax.set_ylabel('RTT (ms)')
    ax.legend(fontsize=8)

if 'C' in dfs:
    ax = fig.add_subplot(gs[2, 1])
    df_c = dfs['C'].copy()
    PHASE_BG = {'normal':'#E8F5E9','burst':'#FFF3E0','idle':'#F3E5F5','burst2':'#E3F2FD'}
    if 'phase' in df_c.columns:
        for phase in df_c['phase'].unique():
            grp = df_c[df_c['phase'] == phase]
            ax.axvspan(grp['timestamp'].min(), grp['timestamp'].max(),
                       alpha=0.2, color=PHASE_BG.get(str(phase),'white'))
    has_data = False
    for col, color, label in [
        ('ue_tx_kbps','#2196F3','UE Uplink'),
        ('ue_rx_kbps','#4CAF50','UE Downlink'),
        ('ogstun_tx_kbps','#FF9800','UPF TX'),
    ]:
        if col in df_c.columns:
            vals = pd.to_numeric(df_c[col], errors='coerce').fillna(0)
            if vals.max() > 0:
                ax.plot(df_c['timestamp'], vals, label=label,
                        linewidth=1.8, alpha=0.9, color=color)
                has_data = True
    if not has_data:
        ax.text(0.5, 0.5, 'KB/s sıfır\nSenaryo C hücresini yeniden çalıştırın',
                transform=ax.transAxes, ha='center', va='center',
                fontsize=10, color='red',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.set_title('Senaryo C — Throughput (KB/s)')
    ax.set_ylabel('KB/s')
    ax.legend(fontsize=9)
    ax.tick_params(axis='x', rotation=20)

plt.savefig('5g_analiz_temel.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Kaydedildi: 5g_analiz_temel.png")


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 15 — GÖRSELLEŞTİRME: 5G SPESİFİK SENARYOLAR  [v2 - DÜZELTİLDİ]  ║
# ╠══════════════════════════════════════════════════════════════════════════════╣
# ║  Plot 1: Handover RTT + HO marker + KPI annotasyon (title'da)              ║
# ║  Plot 2: PDU Session lifecycle + teardown/reestab ms                        ║
# ║  Plot 3: QoS RTT boxplot + medyan annotasyon                               ║
# ║  Plot 4: Slice RTT + URLLC SLA % (title'da)                               ║
# ║  Plot 5: Senaryo C throughput + faz arka planı                             ║
# ║  v2 Düzeltmeleri:                                                           ║
# ║  • NaT/sıfır guard tüm axvline/scatter'larda                              ║
# ║  • Eksik CSV → gri "bulunamadi" mesajı (crash yok)                        ║
# ║  • errors='coerce' tüm sayısal kolonlarda                                  ║
# ║  • "5G-A: ..." özet yorumu hücreden kaldırıldı (SyntaxError düzeltildi)   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

PHASE_COLORS = {
    'PRE_HANDOVER': '#4CAF50', 'HANDOVER':     '#F44336',
    'POST_HANDOVER':'#FF9800', 'STEADY_STATE': '#2196F3',
}
COLORS_SESS = {
    'SESSION_ACTIVE':       '#4CAF50', 'DATA_TRANSFER':    '#8BC34A',
    'SESSION_TEARDOWN':     '#FF5722', 'SESSION_RELEASING':'#FF9800',
    'SESSION_RELEASED':     '#F44336', 'SESSION_ESTABLISHING':'#FFCC02',
    'SESSION_ESTABLISHED':  '#2196F3',
}
PHASE_BG = {
    'normal':'#E8F5E9', 'burst':'#FFF3E0',
    'idle':  '#F3E5F5', 'burst2':'#E3F2FD',
}

fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.5, wspace=0.35)
fig.suptitle('5G AI/ML Lab — 5G Spesifik Senaryo Analizi',
             fontsize=15, fontweight='bold')

# ── Plot 1: Handover RTT ──────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, :])
if 'handover' in dfs:
    df = dfs['handover'].copy()
    df['rtt_ms'] = pd.to_numeric(df['rtt_ms'], errors='coerce').fillna(0)
    df_rtt = df[df['rtt_ms'] > 0]
    if not df_rtt.empty:
        for phase, grp in df_rtt.groupby('phase'):
            ax.scatter(grp['timestamp'], grp['rtt_ms'],
                       label=phase, c=PHASE_COLORS.get(phase,'gray'),
                       s=60, alpha=0.85, zorder=3)
    ho_start_ts = df.loc[df['event'] == 'HO_START', 'timestamp']
    ho_done_ts  = df.loc[df['event'] == 'HO_COMPLETE', 'timestamp']
    if not ho_start_ts.empty and pd.notna(ho_start_ts.min()):
        ax.axvline(ho_start_ts.min(), color='red', linestyle='--',
                   linewidth=2, label='HO Baslangic')
    if not ho_done_ts.empty and pd.notna(ho_done_ts.min()):
        ax.axvline(ho_done_ts.min(), color='green', linestyle='--',
                   linewidth=2, label='HO Tamamlandi')
    pre  = df_rtt[df_rtt['phase'] == 'PRE_HANDOVER']['rtt_ms'].mean()
    post = df_rtt[df_rtt['phase'] == 'POST_HANDOVER']['rtt_ms'].mean()
    ss   = df_rtt[df_rtt['phase'] == 'STEADY_STATE']['rtt_ms'].mean()
    ho_str = ""
    if not ho_start_ts.empty and not ho_done_ts.empty:
        ho_ms = (ho_done_ts.min() - ho_start_ts.min()).total_seconds() * 1000
        ho_str = f" | HO:{ho_ms:.0f}ms"
    kpi = f"Pre:{pre:.3f}ms → Post:{post:.3f}ms → Steady:{ss:.3f}ms{ho_str}"
    ax.set_title(f'Handover Simülasyonu — RTT Zaman Serisi\n{kpi}', fontsize=11)
else:
    ax.text(0.5, 0.5, 'scenario_handover.csv bulunamadi',
            transform=ax.transAxes, ha='center', fontsize=12, color='gray')
    ax.set_title('Handover Simülasyonu', fontsize=12)
ax.set_ylabel('RTT (ms)')
ax.legend(ncol=6, fontsize=8)
ax.tick_params(axis='x', rotation=20)

# ── Plot 2: Session Lifecycle ─────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
if 'session' in dfs:
    df = dfs['session'].copy()
    EVENT_Y = {
        'SESSION_ACTIVE':1.0,'DATA_TRANSFER':0.9,'SESSION_TEARDOWN':0.5,
        'SESSION_RELEASING':0.3,'SESSION_RELEASED':0.1,
        'SESSION_ESTABLISHING':0.3,'SESSION_ESTABLISHED':1.0,
    }
    plotted = set()
    for event, grp in df.groupby('event'):
        y   = EVENT_Y.get(event, 0.5)
        lbl = event if event not in plotted else f'_{event}'
        ax.scatter(grp['timestamp'], [y] * len(grp),
                   label=lbl, c=COLORS_SESS.get(event,'gray'), s=40, alpha=0.8)
        plotted.add(event)
    ax.set_yticks([0.1, 0.3, 0.5, 0.9, 1.0])
    ax.set_yticklabels(['Released','Estab/Releas','Teardown','Data','Active'], fontsize=8)
    td = pd.to_numeric(
        df[df['event'] == 'SESSION_RELEASED']['duration_ms'], errors='coerce').mean()
    es = pd.to_numeric(
        df[df['event'].str.strip() == 'SESSION_ESTABLISHED']['duration_ms'],
        errors='coerce').mean()
    kpi_str = f"  Teardown:{td:.0f}ms  ReEstab:{es:.0f}ms" \
        if not (np.isnan(td) or np.isnan(es)) else ""
    ax.set_title(f'PDU Session Lifecycle (3 Cycle){kpi_str}', fontsize=10)
    ax.legend(fontsize=6, ncol=2, loc='lower right')
else:
    ax.text(0.5, 0.5, 'scenario_session.csv bulunamadi',
            transform=ax.transAxes, ha='center', fontsize=11, color='gray')
    ax.set_title('PDU Session Lifecycle', fontsize=11)
ax.tick_params(axis='x', rotation=30)

# ── Plot 3: QoS RTT Boxplot ───────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 1])
if 'qos' in dfs:
    df = dfs['qos'].copy()
    df['rtt_ms']    = pd.to_numeric(df['rtt_ms'],    errors='coerce').fillna(0)
    df['qos_class'] = df['qos_class'].astype(str)
    df_rtt          = df[df['rtt_ms'] > 0]
    QOS_COLORS      = {'eMBB':'#2196F3','mMTC':'#4CAF50','URLLC':'#FF5722'}
    qos_labels      = [q for q in ['eMBB','mMTC','URLLC']
                       if q in df_rtt['qos_class'].values]
    qos_data        = [df_rtt[df_rtt['qos_class']==q]['rtt_ms'].values for q in qos_labels]
    if qos_data and any(len(d) > 0 for d in qos_data):
        bp = ax.boxplot(qos_data, labels=qos_labels, patch_artist=True,
                        medianprops=dict(color='black', linewidth=2))
        for patch, lbl in zip(bp['boxes'], qos_labels):
            patch.set_facecolor(QOS_COLORS.get(lbl,'gray'))
            patch.set_alpha(0.75)
        ax.axhline(5, color='red', linestyle=':', linewidth=1.5, label='URLLC SLA 5ms')
        ax.legend(fontsize=9)
        for i, (lbl, data) in enumerate(zip(qos_labels, qos_data), 1):
            if len(data) > 0:
                ax.text(i, np.median(data)+0.05, f'{np.median(data):.2f}ms',
                        ha='center', fontsize=8)
    else:
        ax.text(0.5, 0.5, 'QoS verisi yok',
                transform=ax.transAxes, ha='center', fontsize=11, color='gray')
else:
    ax.text(0.5, 0.5, 'scenario_qos.csv bulunamadi',
            transform=ax.transAxes, ha='center', fontsize=11, color='gray')
ax.set_title('QoS Sınıfı RTT Karşılaştırması', fontsize=11)
ax.set_ylabel('RTT (ms)')

# ── Plot 4: Network Slice RTT ─────────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 0])
if 'slice' in dfs:
    df = dfs['slice'].copy()
    df['rtt_ms']     = pd.to_numeric(df['rtt_ms'],     errors='coerce').fillna(0)
    df['slice_type'] = df['slice_type'].astype(str)
    SLICE_COLORS     = {'eMBB':'#2196F3','URLLC':'#F44336','mMTC':'#4CAF50'}
    for stype, grp in df[df['rtt_ms']>0].groupby('slice_type'):
        ax.scatter(grp['timestamp'], grp['rtt_ms'],
                   label=stype, c=SLICE_COLORS.get(stype,'gray'), s=50, alpha=0.75)
    ax.axhline(5, color='red', linestyle=':', linewidth=1.5, label='URLLC SLA 5ms')
    urllc = df[df['slice_type']=='URLLC']
    if not urllc.empty and 'slice_kpi' in urllc.columns:
        sla_pct = (urllc['slice_kpi'].astype(str) == 'SLA_MET').mean() * 100
        ax.set_title(f'Network Slice RTT  (URLLC SLA: {sla_pct:.1f}% met)', fontsize=11)
    else:
        ax.set_title('Network Slice RTT Profilleri', fontsize=11)
    ax.legend(fontsize=9)
else:
    ax.text(0.5, 0.5, 'scenario_slice.csv bulunamadi',
            transform=ax.transAxes, ha='center', fontsize=11, color='gray')
    ax.set_title('Network Slice RTT', fontsize=11)
ax.set_ylabel('RTT (ms)')
ax.tick_params(axis='x', rotation=30)

# ── Plot 5: Senaryo C Throughput ──────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 1])
if 'C' in dfs:
    df = dfs['C'].copy()
    kbps_cols = ['ue_tx_kbps','ue_rx_kbps','ogstun_tx_kbps','ogstun_rx_kbps']
    for col in kbps_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    if 'phase' in df.columns:
        for phase in df['phase'].unique():
            grp = df[df['phase']==phase]
            ax.axvspan(grp['timestamp'].min(), grp['timestamp'].max(),
                       alpha=0.18, color=PHASE_BG.get(str(phase),'white'), zorder=1)
    has_data = False
    for col, color, label in [
        ('ue_tx_kbps',    '#2196F3','UE Uplink'),
        ('ue_rx_kbps',    '#4CAF50','UE Downlink'),
        ('ogstun_tx_kbps','#FF9800','UPF TX'),
    ]:
        if col in df.columns and df[col].max() > 0:
            ax.plot(df['timestamp'], df[col],
                    label=label, linewidth=2, alpha=0.9, color=color, zorder=3)
            has_data = True
    if not has_data:
        ax.text(0.5, 0.5,
                'KB/s değerleri sıfır\nSenaryo C hücresini yeniden çalıştırın',
                transform=ax.transAxes, ha='center', va='center',
                fontsize=10, color='red',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5), zorder=4)
    ax.set_title('Senaryo C — Throughput (KB/s)', fontsize=11)
    ax.set_ylabel('KB/s')
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'scenario_c.csv bulunamadi',
            transform=ax.transAxes, ha='center', fontsize=11, color='gray')
    ax.set_title('Senaryo C — Throughput', fontsize=11)
ax.tick_params(axis='x', rotation=30)

plt.savefig('5g_advanced_analiz.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Kaydedildi: 5g_advanced_analiz.png")


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 16 — GÖRSELLEŞTİRME: MULTI-UE PROFİL ANALİZİ                     ║
# ╠══════════════════════════════════════════════════════════════════════════════╣
# ║  Plot 1: RTT zaman serisi UE bazlı                                          ║
# ║  Plot 2: Profil bazlı RTT boxplot                                           ║
# ║  Plot 3: UE bazlı toplam byte transfer                                      ║
# ║  Plot 4: Inter-arrival dağılımı profil bazlı                               ║
# ║  Plot 5: RTT × Protokol ısı haritası                                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

if 'multi_ue' not in dfs:
    print("⚠️  scenario_multi_ue.csv bulunamadi — Hücre 6'yi calistirin")
else:
    df = dfs['multi_ue'].copy()
    df['rtt_ms'] = pd.to_numeric(df['rtt_ms'], errors='coerce').fillna(0)
    df['bytes']  = pd.to_numeric(df['bytes'],  errors='coerce').fillna(0)
    df['inter_arrival_ms'] = df['timestamp'].diff().dt.total_seconds().mul(1000).fillna(0)

    UE_COLORS      = {'UE1':'#2196F3','UE2':'#4CAF50','UE3':'#FF9800'}
    PROFILE_COLORS = {'mobile_broadband':'#2196F3','iot_sensor':'#4CAF50',
                      'video_stream':'#FF9800'}

    fig = plt.figure(figsize=(18, 14))
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)
    fig.suptitle(
        '5G Multi-UE Profil Analizi\n'
        '(UE1=Mobile Broadband | UE2=IoT Sensor | UE3=Video Stream)',
        fontsize=13, fontweight='bold')

    ax = fig.add_subplot(gs[0, :])
    for ue, grp in df[df['rtt_ms']>0].groupby('ue_id'):
        ax.scatter(grp['timestamp'], grp['rtt_ms'],
                   label=ue, c=UE_COLORS.get(ue,'gray'), s=55, alpha=0.8, zorder=3)
    ax.set_title('RTT Zaman Serisi — UE Bazlı')
    ax.set_ylabel('RTT (ms)')
    ax.legend(ncol=3, fontsize=10)
    ax.tick_params(axis='x', rotation=20)

    ax = fig.add_subplot(gs[1, 0])
    profiles = [p for p in ['mobile_broadband','iot_sensor','video_stream']
                if p in df['profile'].values]
    df_rtt = df[df['rtt_ms']>0]
    bp = ax.boxplot([df_rtt[df_rtt['profile']==p]['rtt_ms'].values for p in profiles],
                    labels=profiles, patch_artist=True)
    for patch, p in zip(bp['boxes'], profiles):
        patch.set_facecolor(PROFILE_COLORS.get(p,'gray'))
        patch.set_alpha(0.75)
    ax.set_title('RTT Dağılımı — Profil Bazlı')
    ax.set_ylabel('RTT (ms)')
    ax.tick_params(axis='x', rotation=15)

    ax = fig.add_subplot(gs[1, 1])
    byte_sum = df.groupby('ue_id')['bytes'].sum() / 1024
    byte_sum.plot(kind='bar', ax=ax,
                  color=[UE_COLORS.get(u,'gray') for u in byte_sum.index],
                  edgecolor='white', alpha=0.85)
    ax.set_title('Toplam Veri Transferi — UE Bazlı')
    ax.set_ylabel('KB')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)
    for bar, val in zip(ax.patches, byte_sum):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.5,
                f'{val:.1f}KB', ha='center', fontsize=9)

    ax = fig.add_subplot(gs[2, 0])
    for profile, grp in df[df['inter_arrival_ms']>0].groupby('profile'):
        grp['inter_arrival_ms'].clip(upper=8000).hist(
            ax=ax, bins=20, alpha=0.6,
            color=PROFILE_COLORS.get(profile,'gray'),
            label=profile, edgecolor='white')
    ax.set_title('Inter-Arrival Time — Profil Bazlı')
    ax.set_xlabel('ms')
    ax.set_ylabel('Frekans')
    ax.legend(fontsize=9)

    ax = fig.add_subplot(gs[2, 1])
    pivot = df.pivot_table(
        index='protocol', columns='ue_id',
        values='rtt_ms', aggfunc='mean').fillna(0)
    sns.heatmap(pivot, ax=ax, annot=True, fmt='.2f',
                cmap='YlOrRd', linewidths=0.5,
                cbar_kws={'label':'Ort RTT (ms)'})
    ax.set_title('Ort RTT — Protokol x UE')
    ax.set_xlabel('')

    plt.savefig('5g_multi_ue_analiz.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Kaydedildi: 5g_multi_ue_analiz.png")